# LAB 05 (v2): HOG-Based Industrial Defect Detection — Rewritten Pipeline

**Pipeline:** `Steel image → Denoise + CLAHE + Resize → HOG descriptor → StandardScaler → Classifier → ACCEPT / REJECT`

**Dataset:** NEU Steel Surface Defect (Kaggle train/valid split) · **Classifiers:** SVM (RBF + Linear), Random Forest, Logistic Regression, k-NN

This is an alternative implementation of the same lab. Compared with the reference version it uses:
* **Grid-based window sampling** (deterministic strided windows, not random crops) to build the binary patch dataset,
* **Preprocessing v2:** Gaussian denoising + CLAHE contrast enhancement before resizing,
* **sklearn Pipelines** (`StandardScaler` + classifier) so scaling is fit only on training data,
* **Image-grouped splitting** for HOG tuning to prevent leakage,
* **Matplotlib `PdfPages`** for the report instead of ReportLab.

> **Dataset note.** NEU-DET contains only *defective* images (6 types with XML boxes). The binary task (0 = Non-Defective, 1 = Defective) is therefore solved at **patch level**:
> * Class 1 = windows whose centre falls inside an annotated defect box,
> * Class 0 = strided background windows that do not intersect any defect box.
> The 6-class problem is solved on whole images in the advanced section.

### Notebook map
| Section | Lab task(s) |
|---|---|
| 1–2. Setup, download, load & inspect | Task 1 |
| 3. Preprocessing v2 + binary patch construction, resize, grayscale | Tasks 2, 3 |
| 4. HOG extraction & visualisation | Tasks 4, 5 |
| 5. Classifier comparison, confusion matrices, metrics | Tasks 6–10 |
| 6. HOG parameter study (cell × orientations) | Task 11 |
| 7. Final model | Tasks 9, 10 |
| 8. Multi-class (6 defect types) | Advanced |
| 9. Robustness (brightness / noise / rotation / blur) | Task 12 |
| 10. Quality-control decision module | Task 13 |
| 11. Webcam / video bonus | Bonus |
| 12. Auto-generated report (PDF + MD) | Report |

## 1. Setup

In [ ]:
# ---------------------------------------------------------------
# LAB 05 v2 — alternative implementation (different code, same lab)
# ---------------------------------------------------------------
import os, re, glob, time, random, warnings, itertools
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import joblib
import matplotlib.pyplot as plt

from skimage.feature import hog
from skimage import exposure as sk_exposure

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)

warnings.filterwarnings("ignore")
RNG = np.random.default_rng(2024)
random.seed(2024); np.random.seed(2024)

# ---------------- configuration (different from v1) ----------------
PATCH   = 96          # crop window size from the 200x200 steel image
RESIZE  = 96          # all patches resized to RESIZE x RESIZE
STRIDE_BG = 48        # background (class 0) windows taken on a grid with this stride
CELLS   = (4, 8, 16)          # required experiment: cell sizes
BINS    = (6, 9, 12)          # required experiment: orientation bins
BASE_CELL, BASE_BIN = 8, 9    # baseline HOG configuration
BLOCK_CELLS = 2               # cells per block (2x2)
MC_RESIZE = 128               # resolution for the whole-image 6-class task

RESULTS = Path("results_v2"); RESULTS.mkdir(exist_ok=True)
LABELS = ["Non-Defective", "Defective"]
print("OpenCV", cv2.__version__, "| numpy", np.__version__, "| sklearn", __import__("sklearn").__version__)

## 2. Dataset download

In [ ]:
LOCAL_PATH = None  # set to a local folder to skip the download

if LOCAL_PATH:
    root = Path(LOCAL_PATH)
else:
    import kagglehub
    root = Path(kagglehub.dataset_download("sovitrath/neu-steel-surface-defect-detect-trainvalid-split"))

print("Dataset root:", root)
files = sorted(root.rglob("*"))
exts = {p.suffix.lower() for p in files if p.is_file()}
print("File extensions found:", exts)
for lvl in sorted({len(p.relative_to(root).parts) for p in files}):
    for d in sorted({p.relative_to(root).parent for p in files if len(p.relative_to(root).parts) == lvl}):
        n = len(list((root / d).glob("*")))
        print("  " * lvl + str(d) + f"/ ({n} entries)")

## 2b. Load & inspect (Task 1)

In [ ]:
DEFECT_TYPES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
IMG_SUFFIX = {".jpg", ".jpeg", ".png", ".bmp"}

def which_split(p: Path):
    tags = {t.lower() for t in p.parts}
    if tags & {"train", "training"}: return "train"
    if tags & {"valid", "val", "validation", "test"}: return "valid"
    return None

def which_class(stem: str):
    key = stem.lower().replace("-", "_")
    for k in DEFECT_TYPES:
        if key.startswith(k): return k
    return re.sub(r"[_-]?\d+$", "", key)

xml_by_stem = {p.stem: p for p in root.rglob("*.xml")}
entries = []
for p in root.rglob("*"):
    if p.suffix.lower() not in IMG_SUFFIX: continue
    entries.append(dict(file=str(p), stem=p.stem, split=which_split(p), defect=which_class(p.stem),
                        xml=str(xml_by_stem.get(p.stem, ""))))
catalog = pd.DataFrame(entries)
assert len(catalog), "No images found – check the dataset path."

if catalog.split.isna().all():                      # fallback split if the dataset has none
    idx_tr, idx_va = train_test_split(catalog.index, test_size=0.2, stratify=catalog.defect, random_state=2024)
    catalog.loc[idx_tr, "split"], catalog.loc[idx_va, "split"] = "train", "valid"
catalog = catalog.dropna(subset=["split"]).reset_index(drop=True)

print(f"images: {len(catalog)} | annotated: {(catalog.xml != '').sum()}")
display(pd.crosstab(catalog.defect, catalog.split, margins=True))

In [ ]:
def read_boxes(xml_file: str):
    # Parse all <bndbox> entries; returns a list of (x1, y1, x2, y2).
    if not xml_file or not os.path.exists(xml_file): return []
    boxes = []
    for obj in ET.parse(xml_file).getroot().iter("object"):
        bb = obj.find("bndbox")
        if bb is None: continue
        boxes.append(tuple(int(float(bb.find(t).text)) for t in ("xmin", "ymin", "xmax", "ymax")))
    return boxes

catalog["boxes"] = catalog.xml.map(read_boxes)
catalog["n_defects"] = catalog.boxes.map(len)

probe = catalog.sample(min(40, len(catalog)), random_state=2024).file
shapes = {cv2.imread(f).shape[:2] for f in probe}
print("sampled image shapes:", shapes, "| mean defects/image:", round(catalog.n_defects.mean(), 2))

# --- gallery: one row per defect type, boxes drawn ---
types = sorted(catalog.defect.unique())
fig, axs = plt.subplots(len(types), 5, figsize=(13, 2.6 * len(types)))
axs = np.atleast_2d(axs)
for r, t in enumerate(types):
    for a, (_, row) in zip(axs[r], catalog[(catalog.defect == t) & (catalog.split == "train")].sample(5, random_state=2024).iterrows()):
        im = cv2.cvtColor(cv2.imread(row.file), cv2.COLOR_BGR2RGB)
        for (x1, y1, x2, y2) in row.boxes: cv2.rectangle(im, (x1, y1), (x2, y2), (255, 50, 50), 2)
        a.imshow(im); a.axis("off")
    axs[r, 0].set_ylabel(t, rotation=0, ha="right", va="center", fontsize=10, fontweight="bold")
fig.suptitle("NEU-DET samples (red boxes = XML annotations)")
fig.tight_layout(); fig.savefig(RESULTS / "fig1_dataset.png", dpi=130); plt.show()

## 3. Preprocessing v2 + binary patch construction (Tasks 2 & 3)

**Preprocessing v2** (different from the reference version):
1. convert to grayscale,
2. Gaussian denoise (`5x5`),
3. CLAHE contrast enhancement,
4. resize to the common `RESIZE × RESIZE`.

**Binary dataset construction (grid-based):**
* Class 1: for every annotated box, the `PATCH × PATCH` window centred on the box centre.
* Class 0: deterministic grid of candidate windows (stride `STRIDE_BG`), kept only if they have **zero** intersection with every box.

In [ ]:
# ---------- preprocessing v2 ----------
def prep_v2(img_bgr, size=RESIZE):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr
    g = cv2.GaussianBlur(g, (5, 5), 0)                       # denoise
    g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)  # local contrast
    return cv2.resize(g, (size, size), interpolation=cv2.INTER_LINEAR)

# ---------- geometry helpers ----------
def center_window(shape, cx, cy, win):
    H, W = shape[:2]; w = min(win, H, W)
    x0 = int(np.clip(cx - w // 2, 0, W - w)); y0 = int(np.clip(cy - w // 2, 0, H - w))
    return x0, y0, w

def intersects(x0, y0, w, boxes):
    for (bx1, by1, bx2, by2) in boxes:
        if min(x0 + w, bx2) - max(x0, bx1) > 0 and min(y0 + w, by2) - max(y0, by1) > 0:
            return True
    return False

# ---------- grid-based binary patch builder ----------
def build_split(frame):
    X, y, info = [], [], []
    for _, row in frame.iterrows():
        img = cv2.imread(row.file); H, W = img.shape[:2]
        if not row.boxes:                                   # no annotation: whole image is defective
            X.append(prep_v2(img)); y.append(1)
            info.append(dict(src=row.file, type=row.defect, kind="defect", x=0, y=0)); continue
        for (x1, y1, x2, y2) in row.boxes:                  # class 1: centred on each defect
            x0, yy, w = center_window(img.shape, (x1 + x2) // 2, (y1 + y2) // 2, PATCH)
            X.append(prep_v2(img[yy:yy + w, x0:x0 + w])); y.append(1)
            info.append(dict(src=row.file, type=row.defect, kind="defect", x=x0, y=yy))
        w = min(PATCH, H, W)                                # class 0: strided background grid
        for yy in range(0, H - w + 1, STRIDE_BG):
            for x0 in range(0, W - w + 1, STRIDE_BG):
                if not intersects(x0, yy, w, row.boxes):
                    X.append(prep_v2(img[yy:yy + w, x0:x0 + w])); y.append(0)
                    info.append(dict(src=row.file, type=row.defect, kind="normal", x=x0, y=yy))
    return np.stack(X).astype(np.uint8), np.asarray(y), pd.DataFrame(info)

t0 = time.time()
Xtr_raw, ytr_raw, info_tr = build_split(catalog[catalog.split == "train"])
Xte_raw, yte_raw, info_te = build_split(catalog[catalog.split == "valid"])
print(f"built in {time.time()-t0:.1f}s")
for tag, X, y in [("TRAIN", Xtr_raw, ytr_raw), ("VALID", Xte_raw, yte_raw)]:
    print(f"{tag}: {X.shape}  normal={int((y==0).sum())}  defective={int((y==1).sum())}")

# --- 80/20 sub-split of TRAIN grouped by source image (used for HOG tuning; test stays untouched) ---
uniq = info_tr.src.unique(); RNG.shuffle(uniq)
cutoff = int(0.8 * len(uniq))
is_sub = info_tr.src.isin(set(uniq[:cutoff])).values
sub_tr, sub_va = np.where(is_sub)[0], np.where(~is_sub)[0]
print(f"sub-train patches: {len(sub_tr)} | tuning-validation patches: {len(sub_va)}")

# --- preview of both classes ---
fig, axs = plt.subplots(2, 10, figsize=(16, 3.6))
for r, lab in enumerate([0, 1]):
    pick = RNG.choice(np.where(ytr_raw == lab)[0], 10, replace=False)
    for a, i in zip(axs[r], pick):
        a.imshow(Xtr_raw[i], cmap="gray"); a.axis("off")
        if lab == 1: a.set_title(info_tr.type[i][:9], fontsize=8)
    axs[r, 0].set_ylabel(LABELS[lab], rotation=90, fontsize=9)
fig.suptitle(f"Preprocessed v2 patches (denoise + CLAHE, {RESIZE}x{RESIZE})")
fig.tight_layout(); fig.savefig(RESULTS / "fig2_patches.png", dpi=130); plt.show()

## 4. HOG extraction & visualisation (Tasks 4 & 5)

In [ ]:
def hog_vec(img_u8, cell=BASE_CELL, nbins=BASE_BIN):
    # HOG feature vector (L2-Hys blocks, sqrt transform).
    return hog(img_u8, orientations=nbins, pixels_per_cell=(cell, cell),
               cells_per_block=(BLOCK_CELLS, BLOCK_CELLS), block_norm="L2-Hys",
               transform_sqrt=True, feature_vector=True)

def hog_batch(imgs, cell=BASE_CELL, nbins=BASE_BIN):
    return np.stack([hog_vec(im, cell, nbins) for im in imgs]).astype(np.float32)

_cache = {}
def features(cell, nbins):
    if (cell, nbins) not in _cache:
        _cache[(cell, nbins)] = (hog_batch(Xtr_raw, cell, nbins), hog_batch(Xte_raw, cell, nbins))
    return _cache[(cell, nbins)]

Ftr, Fte = features(BASE_CELL, BASE_BIN)
print(f"baseline HOG: cell {BASE_CELL}x{BASE_CELL}, {BASE_BIN} bins -> {Ftr.shape[1]} dims | {Ftr.shape[0]} train / {Fte.shape[0]} valid")

# ---------- visualisation: one representative per class ----------
rep_idx = [int(np.where(ytr_raw == 0)[0][0])] + [int(np.where((info_tr.type == t) & (info_tr.kind == "defect"))[0][0]) for t in DEFECT_TYPES]
fig, axs = plt.subplots(2, len(rep_idx), figsize=(2.4 * len(rep_idx), 5))
for j, i in enumerate(rep_idx):
    _, hv = hog(Xtr_raw[i], orientations=BASE_BIN, pixels_per_cell=(BASE_CELL, BASE_CELL),
                cells_per_block=(BLOCK_CELLS, BLOCK_CELLS), block_norm="L2-Hys",
                transform_sqrt=True, visualize=True)
    axs[0, j].imshow(Xtr_raw[i], cmap="gray"); axs[0, j].set_title("normal" if j == 0 else info_tr.type[i], fontsize=9)
    axs[1, j].imshow(sk_exposure.rescale_intensity(hv, in_range=(0, hv.max() + 1e-9)), cmap="gray")
    for r in range(2): axs[r, j].axis("off")
axs[0, 0].set_ylabel("patch", rotation=0, ha="right"); axs[1, 0].set_ylabel("HOG", rotation=0, ha="right")
fig.suptitle(f"HOG visualisation (cell {BASE_CELL}, {BASE_BIN} bins)"); fig.tight_layout()
fig.savefig(RESULTS / "fig3_hog.png", dpi=130); plt.show()

# ---------- effect of HOG parameters ----------
probe = rep_idx[1]
fig, axs = plt.subplots(len(BINS), len(CELLS), figsize=(8.5, 8.5))
for r, nb in enumerate(BINS):
    for c, cs in enumerate(CELLS):
        _, hv = hog(Xtr_raw[probe], orientations=nb, pixels_per_cell=(cs, cs),
                    cells_per_block=(BLOCK_CELLS, BLOCK_CELLS), block_norm="L2-Hys",
                    transform_sqrt=True, visualize=True)
        axs[r, c].imshow(sk_exposure.rescale_intensity(hv, in_range=(0, hv.max() + 1e-9)), cmap="gray")
        axs[r, c].set_title(f"cell {cs}² | {nb} bins", fontsize=9); axs[r, c].axis("off")
fig.suptitle("HOG representation vs parameters (defective patch)"); fig.tight_layout()
fig.savefig(RESULTS / "fig4_hog_params.png", dpi=130); plt.show()

## 5. Classifier comparison (Tasks 6–10)

In [ ]:
def make_pipelines():
    # Scaler + classifier inside a sklearn Pipeline (scaling never sees test data).
    return {
        "SVM-RBF":  Pipeline([("sc", StandardScaler()), ("clf", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=2024))]),
        "SVM-Lin":  Pipeline([("sc", StandardScaler()), ("clf", LinearSVC(C=0.1, class_weight="balanced", max_iter=20000, random_state=2024))]),
        "RF":       Pipeline([("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=2024))]),
        "LogReg":   Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(C=10, max_iter=5000, class_weight="balanced", random_state=2024))]),
        "kNN":      Pipeline([("sc", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=7))]),
    }

def score(y_true, y_pred, avg="binary"):
    return dict(Accuracy=accuracy_score(y_true, y_pred),
                Precision=precision_score(y_true, y_pred, average=avg, zero_division=0),
                Recall=recall_score(y_true, y_pred, average=avg, zero_division=0),
                F1=f1_score(y_true, y_pred, average=avg, zero_division=0))

fitted, pred_store, cmp_rows = {}, {}, []
for name, pipe in make_pipelines().items():
    t0 = time.time(); pipe.fit(Ftr, ytr_raw); fit_s = time.time() - t0
    t0 = time.time(); p = pipe.predict(Fte); ms = (time.time() - t0) / len(Fte) * 1000
    fitted[name], pred_store[name] = pipe, p
    cmp_rows.append(dict(Classifier=name, **score(yte_raw, p), Fit_s=fit_s, Ms_per_patch=ms))
cmp_tbl = pd.DataFrame(cmp_rows).set_index("Classifier")
display(cmp_tbl.style.format("{:.4f}").highlight_max(subset=["Accuracy", "Precision", "Recall", "F1"], color="#cdeac0"))

ax = cmp_tbl[["Accuracy", "Precision", "Recall", "F1"]].plot.bar(rot=0, figsize=(10, 4), ylim=(0, 1.05))
ax.set_title(f"Classifier comparison — baseline HOG (cell {BASE_CELL}², {BASE_BIN} bins)")
ax.legend(ncol=4, loc="lower right"); plt.tight_layout(); plt.savefig(RESULTS / "fig5_compare.png", dpi=130); plt.show()

In [ ]:
# ---------- confusion matrices + per-class reports ----------
fig, axs = plt.subplots(1, len(pred_store), figsize=(4 * len(pred_store), 3.8))
for a, (name, p) in zip(np.atleast_1d(axs), pred_store.items()):
    ConfusionMatrixDisplay(confusion_matrix(yte_raw, p), display_labels=LABELS).plot(
        ax=a, cmap="Greens", colorbar=False, values_format="d")
    a.set_title(name); a.tick_params(axis="x", labelrotation=15)
fig.tight_layout(); fig.savefig(RESULTS / "fig6_confmats.png", dpi=130); plt.show()

for name, p in pred_store.items():
    print("=" * 58, f"\n{name}\n" + "=" * 58)
    print(classification_report(yte_raw, p, target_names=LABELS, digits=4))

## 6. HOG parameter study (Task 11)

In [ ]:
tune_rows = []
for cs, nb in itertools.product(CELLS, BINS):
    A, B = features(cs, nb)
    for name in ["SVM-RBF", "RF"]:
        pipe = make_pipelines()[name]
        t0 = time.time(); pipe.fit(A[sub_tr], ytr_raw[sub_tr]); ft = time.time() - t0
        m = score(ytr_raw[sub_va], pipe.predict(A[sub_va]))
        tune_rows.append(dict(Model=name, Cell=f"{cs}x{cs}", Bins=nb, Dims=A.shape[1], **m, Fit_s=ft))
        print(f"{name:8s} cell {cs:>2}² bins {nb:>2} -> F1 {m['F1']:.4f} acc {m['Accuracy']:.4f} ({ft:.0f}s)")
tune = pd.DataFrame(tune_rows)
display(tune.style.format({"Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}", "Fit_s": "{:.0f}"}))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for a, mdl in zip(axs, ["SVM-RBF", "RF"]):
    piv = tune[tune.Model == mdl].pivot(index="Cell", columns="Bins", values="F1").reindex([f"{c}x{c}" for c in CELLS])
    im = a.imshow(piv.values, cmap="magma", aspect="auto", vmin=piv.values.min() - 0.02, vmax=piv.values.max() + 0.02)
    a.set_title(f"{mdl} — tuning F1"); a.set_xticks(range(3), piv.columns); a.set_yticks(range(3), piv.index)
    a.set_xlabel("orientation bins"); a.set_ylabel("cell size")
    for (i, j), v in np.ndenumerate(piv.values): a.text(j, i, f"{v:.3f}", ha="center", va="center", color="w")
    plt.colorbar(im, ax=a)
fig.tight_layout(); fig.savefig(RESULTS / "fig7_grid.png", dpi=130); plt.show()

top = tune[tune.Model == "SVM-RBF"].sort_values(["F1", "Accuracy"], ascending=False).iloc[0]
BEST_CELL, BEST_BIN = int(top.Cell.split("x")[0]), int(top.Bins)
print(f"\nSelected HOG configuration (SVM tuning F1): cell {BEST_CELL}², {BEST_BIN} bins")

## 7. Final model (Tasks 9 & 10)

In [ ]:
Ftr_b, Fte_b = features(BEST_CELL, BEST_BIN)

final_model = Pipeline([
    ("sc", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced",
                probability=True, random_state=2024))])
final_model.fit(Ftr_b, ytr_raw)
y_hat  = final_model.predict(Fte_b)
y_prob = final_model.predict_proba(Fte_b)[:, 1]

fin = score(yte_raw, y_hat); fin["ROC-AUC"] = roc_auc_score(yte_raw, y_prob)
print(f"FINAL: HOG (cell {BEST_CELL}², {BEST_BIN} bins) + SVM-RBF")
for k, v in fin.items(): print(f"  {k:9s}: {v:.4f}")
print("\n" + classification_report(yte_raw, y_hat, target_names=LABELS, digits=4))

cmat = confusion_matrix(yte_raw, y_hat)
fig, ax = plt.subplots(figsize=(4.6, 4))
ConfusionMatrixDisplay(cmat, display_labels=LABELS).plot(ax=ax, cmap="Greens", colorbar=False, values_format="d")
ax.set_title("Final model — test confusion matrix"); plt.tight_layout()
plt.savefig(RESULTS / "fig8_conf_final.png", dpi=130); plt.show()
tn, fp, fn, tp = cmat.ravel()
print(f"TN={tn} FP={fp} (good rejected) | FN={fn} (defect missed) TP={tp}")

# companion model for the robustness study
final_rf = make_pipelines()["RF"].fit(Ftr_b, ytr_raw)
print("RF at same HOG config:", {k: round(v, 4) for k, v in score(yte_raw, final_rf.predict(Fte_b)).items()})

## 8. Advanced: 6-class defect-type classification (whole images)

In [ ]:
tr_cat, te_cat = catalog[catalog.split == "train"], catalog[catalog.split == "valid"]
id2type = sorted(catalog.defect.unique()); type2id = {t: i for i, t in enumerate(id2type)}
Fmc_tr = hog_batch([prep_v2(cv2.imread(f), MC_RESIZE) for f in tr_cat.file], BEST_CELL * 2, BEST_BIN)
Fmc_te = hog_batch([prep_v2(cv2.imread(f), MC_RESIZE) for f in te_cat.file], BEST_CELL * 2, BEST_BIN)
ymc_tr = tr_cat.defect.map(type2id).values; ymc_te = te_cat.defect.map(type2id).values
print("multi-class HOG:", Fmc_tr.shape, Fmc_te.shape)

mc_rows, mc_pred = [], {}
for name in ["SVM-RBF", "RF"]:
    pipe = make_pipelines()[name].fit(Fmc_tr, ymc_tr)
    p = pipe.predict(Fmc_te); mc_pred[name] = p
    mc_rows.append(dict(Model=name, **score(ymc_te, p, avg="macro")))
mc_tbl = pd.DataFrame(mc_rows).set_index("Model")
mc_tbl.columns = ["Accuracy", "MacroP", "MacroR", "MacroF1"]
display(mc_tbl.style.format("{:.4f}"))
best_mc = mc_tbl.MacroF1.idxmax()
print(f"best 6-class model: {best_mc}\n")
print(classification_report(ymc_te, mc_pred[best_mc], target_names=id2type, digits=4))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(confusion_matrix(ymc_te, mc_pred[best_mc]), display_labels=id2type).plot(
    ax=ax, cmap="Greens", colorbar=False, xticks_rotation=40)
ax.set_title(f"6-class confusion matrix — {best_mc}"); plt.tight_layout()
plt.savefig(RESULTS / "fig9_conf_mc.png", dpi=130); plt.show()

## 9. Robustness experiments (Task 12)

Models are trained **once on clean patches** and evaluated on corrupted validation patches: brightness (±), Gaussian noise, rotation, blur — three severities each. The perturbed sets are cached so all models see identical images.

In [ ]:
def corr_brightness(f, beta):  return lambda im: cv2.convertScaleAbs(im, alpha=f, beta=beta)
def corr_noise(sigma):
    gen = np.random.default_rng(99)
    return lambda im: np.clip(im.astype(np.float32) + gen.normal(0, sigma, im.shape), 0, 255).astype(np.uint8)
def corr_rot(deg):
    def fn(im):
        h, w = im.shape; M = cv2.getRotationMatrix2D((w / 2, h / 2), deg, 1.0)
        return cv2.warpAffine(im, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    return fn
def corr_blur(k): return lambda im: cv2.GaussianBlur(im, (k, k), 0)

PERTURB = {
    "bright x0.6": ("brightness", corr_brightness(0.6, 0)), "bright x1.4": ("brightness", corr_brightness(1.4, 0)),
    "bright +40":  ("brightness", corr_brightness(1.0, 40)),
    "noise s=5":   ("noise", corr_noise(5)),  "noise s=15": ("noise", corr_noise(15)),  "noise s=30": ("noise", corr_noise(30)),
    "rot 5°":      ("rotation", corr_rot(5)), "rot 15°":    ("rotation", corr_rot(15)), "rot 30°":    ("rotation", corr_rot(30)),
    "blur k=3":    ("blur", corr_blur(3)),   "blur k=5":   ("blur", corr_blur(5)),   "blur k=9":   ("blur", corr_blur(9)),
}
corrupted = {k: np.stack([fn(im) for im in Xte_raw]) for k, (_, fn) in PERTURB.items()}

def evaluate(model, imgs):
    return score(yte_raw, model.predict(hog_batch(imgs, BEST_CELL, BEST_BIN)))

def robustness_frame(models):
    rows = []
    clean = {n: evaluate(m, Xte_raw) for n, m in models.items()}
    for n, m in models.items():
        rows.append(dict(Condition="clean", Category="-", Model=n, Acc=clean[n]["Accuracy"], F1=clean[n]["F1"], dF1=0.0))
    for cname, (cat, _) in PERTURB.items():
        for n, m in models.items():
            r = evaluate(m, corrupted[cname])
            rows.append(dict(Condition=cname, Category=cat, Model=n, Acc=r["Accuracy"], F1=r["F1"],
                             dF1=r["F1"] - clean[n]["F1"]))
    return pd.DataFrame(rows)

rob = robustness_frame({"HOG+SVM": final_model, "HOG+RF": final_rf})
view = rob.pivot_table(index=["Category", "Condition"], columns="Model", values=["Acc", "F1", "dF1"], sort=False)
display(view.style.format("{:.4f}", subset=[c for c in view.columns if c[0] != "dF1"]).format("{:+.4f}", subset=[c for c in view.columns if c[0] == "dF1"]))

In [ ]:
# ---------- corrupted examples ----------
keys = ["bright x0.6", "bright x1.4", "noise s=15", "noise s=30", "rot 15°", "rot 30°", "blur k=5", "blur k=9"]
i0 = int(np.where(yte_raw == 1)[0][4])
fig, axs = plt.subplots(1, len(keys) + 1, figsize=(17, 2.3))
axs[0].imshow(Xte_raw[i0], cmap="gray", vmin=0, vmax=255); axs[0].set_title("clean", fontsize=8)
for a, k in zip(axs[1:], keys): a.imshow(corrupted[k][i0], cmap="gray", vmin=0, vmax=255); a.set_title(k, fontsize=8)
for a in axs: a.axis("off")
fig.tight_layout(); fig.savefig(RESULTS / "fig10_corrupt.png", dpi=130); plt.show()

# ---------- bar chart + per-category summary ----------
sub = rob[rob.Condition != "clean"]
fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for a, met in zip(axs, ["Acc", "F1"]):
    pv = sub.pivot(index="Condition", columns="Model", values=met).reindex(list(PERTURB))
    pv.plot.barh(ax=a); a.invert_yaxis(); a.set_xlim(0, 1); a.set_title(f"{met} under corruption")
    for mdl, col in zip(["HOG+SVM", "HOG+RF"], ["C0", "C1"]):
        a.axvline(rob[(rob.Condition == "clean") & (rob.Model == mdl)][met].iloc[0], color=col, ls="--", lw=1)
fig.suptitle("Robustness (dashed = clean baseline)"); fig.tight_layout()
fig.savefig(RESULTS / "fig11_robust.png", dpi=130); plt.show()
display(sub.groupby(["Category", "Model"]).dF1.mean().unstack().style.format("{:+.4f}"))

In [ ]:
# ---------- does augmented training help? ----------
aug_rng = np.random.default_rng(7)
fns = [fn for _, fn in PERTURB.values()]
X_aug = np.stack([fns[aug_rng.integers(len(fns))](im) for im in Xtr_raw])
Ftr_aug = np.vstack([Ftr_b, hog_batch(X_aug, BEST_CELL, BEST_BIN)])
ytr_aug = np.concatenate([ytr_raw, ytr_raw])

svm_aug = Pipeline([("sc", StandardScaler()), ("clf", SVC(kernel="rbf", C=10, gamma="scale",
             class_weight="balanced", random_state=2024))]).fit(Ftr_aug, ytr_aug)
rob_aug = robustness_frame({"aug-SVM": svm_aug})
gain_rows = []
for cname in ["clean"] + list(PERTURB):
    base = rob[(rob.Condition == cname) & (rob.Model == "HOG+SVM")].F1.iloc[0]
    augv = rob_aug[rob_aug.Condition == cname].F1.iloc[0]
    gain_rows.append(dict(Condition=cname, F1_clean_trained=base, F1_aug_trained=augv, Gain=augv - base))
gain_tbl = pd.DataFrame(gain_rows).set_index("Condition")
display(gain_tbl.style.format("{:.4f}", subset=["F1_clean_trained", "F1_aug_trained"]).format("{:+.4f}", subset=["Gain"]))

## 10. Quality-control decision module (Task 13)

`inspect()` classifies one patch; `inspect_product()` scans a **whole product image** with a sliding window
(`PATCH` window, stride 40) and rejects the product when at least `MIN_HITS` windows exceed `P_REJECT`.
The model bundle is saved to `results_v2/defect_model_v2.joblib`.

In [ ]:
joblib.dump(dict(model=final_model, cell=BEST_CELL, bins=BEST_BIN, resize=RESIZE, win=PATCH),
            RESULTS / "defect_model_v2.joblib")

P_REJECT = 0.5     # window probability threshold
MIN_HITS = 2       # defective windows needed to reject a product
STRIDE_PROD = 40

def verdict(p):
    return ("DEFECTIVE", "REJECT") if p >= P_REJECT else ("NON-DEFECTIVE", "ACCEPT")

def report(res):
    print("== QUALITY CONTROL ==")
    print(f"  prediction : {res['label']}")
    print(f"  confidence : {res['confidence']:.1f}%")
    print(f"  action     : {res['action']} PRODUCT")

def inspect(patch_u8):
    p = final_model.predict_proba(hog_vec(patch_u8, BEST_CELL, BEST_BIN)[None])[0, 1]
    lab, act = verdict(p)
    res = dict(label=lab, confidence=100 * p if lab == "DEFECTIVE" else 100 * (1 - p), action=act, p_defect=p)
    report(res); return res

def product_scan(img_bgr, stride=STRIDE_PROD, verbose=False):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (5, 5), 0)
    g = cv2.createCLAHE(2.0, (8, 8)).apply(g)
    H, W = g.shape; w = min(PATCH, H, W)
    xy = [(x, y) for y in range(0, H - w + 1, stride) for x in range(0, W - w + 1, stride)]
    F = np.stack([hog_vec(cv2.resize(g[y:y + w, x:x + w], (RESIZE, RESIZE)), BEST_CELL, BEST_BIN) for x, y in xy])
    probs = final_model.predict_proba(F)[:, 1]
    hits = [(x, y, w, p) for (x, y), p in zip(xy, probs) if p >= P_REJECT]
    if len(hits) >= MIN_HITS:
        res = dict(label="DEFECTIVE", confidence=100 * float(np.mean([h[3] for h in hits])), action="REJECT", hits=hits)
    else:
        res = dict(label="NON-DEFECTIVE", confidence=100 * float(1 - probs.mean()), action="ACCEPT", hits=hits)
    if verbose: report(res)
    return res

def inspect_product(img_bgr):
    res = product_scan(img_bgr, verbose=True); return res

# ---- demo: single patches ----
for lab in (1, 0):
    i = int(np.where(yte_raw == lab)[0][7])
    print(f"\n-- patch #{i} (true: {LABELS[lab]}) --"); inspect(Xte_raw[i])

# ---- demo: whole product images ----
demo = te_cat.sample(4, random_state=2024)
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
for a, (_, row) in zip(axs, demo.iterrows()):
    img = cv2.imread(row.file)
    print(f"\n-- {Path(row.file).name} (true type: {row.defect}) --")
    r = inspect_product(img)
    vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for (x, y, w, _) in r["hits"]: cv2.rectangle(vis, (x, y), (x + w, y + w), (255, 60, 60), 2)
    a.imshow(vis); a.axis("off")
    a.set_title(f"{r['label']} {r['confidence']:.0f}%\n{r['action']}", fontsize=9,
                color="crimson" if r["label"] == "DEFECTIVE" else "seagreen")
fig.tight_layout(); fig.savefig(RESULTS / "fig12_qc.png", dpi=130); plt.show()

# ---- image-level reject rate + latency ----
sample = te_cat.sample(min(100, len(te_cat)), random_state=2024)
t0 = time.time(); rej = [product_scan(cv2.imread(f))["action"] == "REJECT" for f in sample.file]
img_ms = (time.time() - t0) / len(sample) * 1000
IMG_RECALL = float(np.mean(rej))
t0 = time.time()
for i in range(100): final_model.predict(Fte_b[i:i + 1])
patch_ms = (time.time() - t0) / 100 * 1000
print(f"\nimage-level reject rate on {len(sample)} defective validation images: {100*IMG_RECALL:.1f}%")
print(f"latency: {patch_ms:.2f} ms/patch | {img_ms:.0f} ms/image (CPU)")

## 11. Bonus: webcam / video prototype

In [ ]:
def frame_verdict(frame_bgr, ref=200, stride=60):
    h, w = frame_bgr.shape[:2]; s = ref / min(h, w)
    scaled = cv2.resize(frame_bgr, (max(PATCH, int(w * s)), max(PATCH, int(h * s))), interpolation=cv2.INTER_AREA)
    g = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.GaussianBlur(cv2.cvtColor(scaled, cv2.COLOR_BGR2GRAY), (5, 5), 0))
    H, W = g.shape; w = min(PATCH, H, W)
    xy = [(x, y) for y in range(0, H - w + 1, stride) for x in range(0, W - w + 1, stride)]
    F = np.stack([hog_vec(cv2.resize(g[y:y + w, x:x + w], (RESIZE, RESIZE)), BEST_CELL, BEST_BIN) for x, y in xy])
    probs = final_model.predict_proba(F)[:, 1]
    hits = [(x, y, w, p) for (x, y), p in zip(xy, probs) if p >= P_REJECT]
    if len(hits) >= MIN_HITS:
        return dict(status="DEFECTIVE", conf=100 * float(np.mean([h[3] for h in hits])), hits=hits, scale=s)
    return dict(status="PASS", conf=100 * float(1 - probs.mean()), hits=[], scale=s)

def draw_overlay(frame, r, fps=None):
    out = frame.copy(); bad = r["status"] == "DEFECTIVE"; col = (0, 0, 255) if bad else (0, 160, 0)
    for (x, y, w, _) in r["hits"]:
        cv2.rectangle(out, (int(x / r["scale"]), int(y / r["scale"])),
                      (int((x + w) / r["scale"]), int((y + w) / r["scale"])), (0, 0, 255), 2)
    cv2.rectangle(out, (0, 0), (out.shape[1], 34), col, -1)
    txt = f"{r['status']} {r['conf']:.0f}%" + (f" {fps:.1f}FPS" if fps else "")
    cv2.putText(out, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    return out

def run_camera(source=0):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened(): print("cannot open", source); return
    prev = time.time()
    while True:
        ok, fr = cap.read()
        if not ok: break
        r = frame_verdict(fr); fps = 1 / max(time.time() - prev, 1e-6); prev = time.time()
        cv2.imshow("HOG QC v2", draw_overlay(fr, r, fps))
        if cv2.waitKey(1) & 0xFF == ord("q"): break
    cap.release(); cv2.destroyAllWindows()

# ---- simulated camera stream (no webcam needed): jittered validation frames ----
sim = te_cat.sample(6, random_state=11); jr = np.random.default_rng(5)
fig, axs = plt.subplots(2, 3, figsize=(13, 8)); t0 = time.time()
for a, (_, row) in zip(axs.ravel(), sim.iterrows()):
    fr = cv2.imread(row.file).astype(np.float32)
    fr = np.clip(fr * jr.uniform(0.9, 1.1) + jr.normal(0, 4, fr.shape), 0, 255).astype(np.uint8)
    r = frame_verdict(fr)
    a.imshow(cv2.cvtColor(draw_overlay(fr, r), cv2.COLOR_BGR2RGB)); a.axis("off")
    a.set_title(f"true: {row.defect}", fontsize=9)
fig.tight_layout(); fig.savefig(RESULTS / "fig13_stream.png", dpi=130); plt.show()
print(f"simulated throughput: {6 / (time.time() - t0):.1f} fps (CPU)")
# run_camera(0)  # locally, press q to quit

In [ ]:
# ---------- standalone real-time script ----------
script = '''
# Real-time HOG steel-surface inspection. Usage: python qc_realtime_v2.py [source]
import sys, time, cv2, joblib, numpy as np
from skimage.feature import hog

B = joblib.load("defect_model_v2.joblib")
M, CELL, BINS, SIZE, WIN = B["model"], B["cell"], B["bins"], B["resize"], B["win"]
THR, MIN_HITS, STRIDE, REF = 0.5, 2, 60, 200

def feat(g):
    g = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.GaussianBlur(g, (5, 5), 0))
    g = cv2.resize(g, (SIZE, SIZE))
    return hog(g, orientations=BINS, pixels_per_cell=(CELL, CELL), cells_per_block=(2, 2),
               block_norm="L2-Hys", transform_sqrt=True, feature_vector=True)

def scan(frame):
    h, w = frame.shape[:2]; s = REF / min(h, w)
    g = cv2.cvtColor(cv2.resize(frame, (max(WIN, int(w * s)), max(WIN, int(h * s))), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    H, W = g.shape
    xy = [(x, y) for y in range(0, H - WIN + 1, STRIDE) for x in range(0, W - WIN + 1, STRIDE)]
    p = M.predict_proba(np.stack([feat(g[y:y+WIN, x:x+WIN]) for x, y in xy]))[:, 1]
    hits = [(x, y) for (x, y), v in zip(xy, p) if v >= THR]
    return ("DEFECTIVE", 100 * float(p[p >= THR].mean())) if len(hits) >= MIN_HITS else ("PASS", 100 * float(1 - p.mean())), hits, s

src = sys.argv[1] if len(sys.argv) > 1 else 0
cap = cv2.VideoCapture(int(src) if str(src).isdigit() else src)
prev = time.time()
while cap.isOpened():
    ok, f = cap.read()
    if not ok: break
    (status, conf), hits, s = scan(f)
    col = (0, 0, 255) if status == "DEFECTIVE" else (0, 160, 0)
    for x, y in hits:
        cv2.rectangle(f, (int(x/s), int(y/s)), (int((x+WIN)/s), int((y+WIN)/s)), (0, 0, 255), 2)
    cv2.rectangle(f, (0, 0), (f.shape[1], 34), col, -1)
    cv2.putText(f, f"{status} {conf:.0f}% {1/(time.time()-prev):.1f}FPS", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2)
    prev = time.time(); cv2.imshow("HOG QC v2 (q quits)", f)
    if cv2.waitKey(1) & 0xFF == ord("q"): break
cap.release(); cv2.destroyAllWindows()
'''
(RESULTS / "qc_realtime_v2.py").write_text(script.lstrip())
print("saved", RESULTS / "qc_realtime_v2.py")

## 12. Auto-generated report (PDF + Markdown)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

# ---- pull numbers from this run ----
n_tr, n_va = len(tr_cat), len(te_cat)
svm_grid = tune[tune.Model == "SVM-RBF"]
g_best, g_worst = svm_grid.loc[svm_grid.F1.idxmax()], svm_grid.loc[svm_grid.F1.idxmin()]
best_clf = cmp_tbl.F1.idxmax()
svm_rob = rob[(rob.Model == "HOG+SVM") & (rob.Condition != "clean")]
worst_c = svm_rob.loc[svm_rob.dF1.idxmin()]; mild_c = svm_rob.loc[svm_rob.dF1.idxmax()]
cat_mean = svm_rob.groupby("Category").dF1.mean()
mc_row = mc_tbl.loc[best_mc]
aug_mean_gain = gain_tbl.drop("clean").Gain.mean()

def tbl_md(d):
    h = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
    return h + "\n".join("| " + " | ".join(f"{v:.4f}" if isinstance(v, float) else str(v) for v in r) + " |" for r in d.values)

SEC = []
SEC.append(("1. Introduction",
    "Automated visual inspection detects surface defects faster and more consistently than manual checks. "
    "This lab implements a steel-surface defect detector: Image > Preprocessing > HOG > Classifier > Defective / Non-Defective decision."))
SEC.append(("2. Industrial Application",
    "In-line quality control of hot-rolled steel: defective strips are rejected or reworked before shipping. "
    "Lightweight HOG + SVM inference is suitable for edge devices mounted next to the line camera."))
SEC.append(("3. Dataset Description",
    f"NEU steel surface defect database (Kaggle train/valid split): {n_tr} training and {n_va} validation images (200x200 px), "
    f"six defect types ({', '.join(id2type)}) with XML bounding boxes. No defect-free images exist, so Class 1 patches are centred on "
    f"annotated defects and Class 0 patches are strided background windows with zero box overlap. Result: "
    f"{int((ytr_raw==0).sum())}/{int((ytr_raw==1).sum())} normal/defective training patches and "
    f"{int((yte_raw==0).sum())}/{int((yte_raw==1).sum())} validation patches. The 6-class task uses whole images."))
SEC.append(("4. Image Preprocessing",
    f"Grayscale -> Gaussian denoise -> CLAHE -> resize to {RESIZE}x{RESIZE}. Local contrast enhancement (CLAHE) makes faint "
    f"defects more visible than plain grayscale."))
SEC.append(("5. HOG Feature Extraction",
    f"2x2-cell blocks, L2-Hys normalisation, sqrt transform. Baseline: {BASE_CELL}x{BASE_CELL} cells, {BASE_BIN} bins. "
    f"Selected: {BEST_CELL}x{BEST_CELL} cells, {BEST_BIN} bins -> {Ftr_b.shape[1]}-dim vectors. Small cells capture fine crazing texture; "
    f"large cells capture coarse shape."))
SEC.append(("6. Classification Methodology",
    "Pipelines of StandardScaler + classifier (RBF-SVM C=10, linear SVM C=0.1, Random Forest 300 trees, Logistic Regression C=10, k-NN k=7). "
    "Balanced class weights for the two SVMs/LogReg/RF. Positive class = Defective."))
SEC.append(("7. Experimental Setup",
    "Classifier comparison at baseline HOG on the validation split. HOG study: cell sizes 4/8/16, bins 6/9/12 with SVM-RBF and RF, "
    "tuned on an 80/20 image-grouped split of TRAIN only; best config retrained on all training patches. "
    "Metrics: accuracy, precision, recall, F1 (Defective = positive), confusion matrices."))
SEC.append(("8. Results",
    f"Best baseline classifier by F1: {best_clf}. Final model: HOG ({BEST_CELL}² cells, {BEST_BIN} bins) + SVM-RBF -> "
    f"accuracy {fin['Accuracy']:.4f}, precision {fin['Precision']:.4f}, recall {fin['Recall']:.4f}, F1 {fin['F1']:.4f}, "
    f"ROC-AUC {fin['ROC-AUC']:.4f}. Confusion: TN={tn} FP={fp} FN={fn} TP={tp}. "
    f"HOG study range (SVM tuning F1): {svm_grid.F1.min():.3f} - {svm_grid.F1.max():.3f} "
    f"(best cell {g_best.Cell}/{int(g_best.Bins)} bins, worst cell {g_worst.Cell}/{int(g_worst.Bins)} bins). "
    f"6-class: best model {best_mc}, accuracy {mc_row.Accuracy:.4f}, macro-F1 {mc_row.MacroF1:.4f}."))
SEC.append(("9. Robustness Analysis",
    f"Worst corruption: '{worst_c.Condition}' (dF1 {worst_c.dF1:+.4f}); mildest: '{mild_c.Condition}' (dF1 {mild_c.dF1:+.4f}). "
    f"Mean dF1 by category: " + ", ".join(f"{k} {v:+.3f}" for k, v in cat_mean.items()) +
    f". Augmented training changed mean F1 by {aug_mean_gain:+.4f} over corrupted conditions."))
SEC.append(("10. Industrial Deployment Discussion",
    f"The module outputs PREDICTION / CONFIDENCE / ACTION. Whole images are scanned with a sliding window and rejected when >= {MIN_HITS} "
    f"windows exceed p={P_REJECT}; image-level reject rate on defective validation images: {100*IMG_RECALL:.1f}%. "
    f"Latency {patch_ms:.1f} ms/patch and {img_ms:.0f} ms/image on CPU - real-time capable. In production: tune the threshold by the cost "
    f"of false rejects vs false accepts, fix lighting/camera pose, route low-confidence items to humans."))
SEC.append(("11. Limitations",
    "(i) Non-Defective class is synthesised from background of defective images - real clean steel may differ; false-reject rate likely "
    "under-estimated. (ii) Small, lab-quality, single-camera images. (iii) HOG is not rotation invariant and degrades under heavy noise/blur. "
    "(iv) Fixed-size windows struggle with very large defects. (v) One train/valid split only."))
SEC.append(("12. Conclusion",
    f"Complete HOG inspection pipeline implemented: F1 {fin['F1']:.3f} (binary), macro-F1 {mc_row.MacroF1:.3f} (6-class). "
    f"HOG parameters mattered ({svm_grid.F1.min():.3f}-{svm_grid.F1.max():.3f} tuning F1). Robustness is corruption-dependent; "
    f"augmentation is a practical remedy. The ACCEPT/REJECT module is fast enough for real-time use."))

tables = {
    "8. Results": [("Classifier comparison (baseline HOG)", cmp_tbl.reset_index()),
                   ("HOG parameter study (SVM tuning)", svm_grid[["Cell", "Bins", "Dims", "Accuracy", "F1"]]),
                   ("Robustness of final SVM", rob[rob.Model == "HOG+SVM"][["Condition", "Acc", "F1", "dF1"]])],
    "9. Robustness Analysis": [("Augmented vs clean training", gain_tbl.reset_index())],
}
images = {
    "4. Image Preprocessing": [("fig2_patches.png", "Fig.1 preprocessed patches")],
    "5. HOG Feature Extraction": [("fig3_hog.png", "Fig.2 HOG visualisation"), ("fig4_hog_params.png", "Fig.3 HOG vs parameters")],
    "8. Results": [("fig5_compare.png", "Fig.4 classifier comparison"), ("fig7_grid.png", "Fig.5 tuning heat-maps"),
                   ("fig8_conf_final.png", "Fig.6 final confusion matrix"), ("fig9_conf_mc.png", "Fig.7 6-class confusion matrix")],
    "9. Robustness Analysis": [("fig11_robust.png", "Fig.8 robustness")],
    "10. Industrial Deployment Discussion": [("fig12_qc.png", "Fig.9 QC module demo")],
}

# ---- markdown version ----
md_lines = ["# LAB 05 v2 — HOG-Based Industrial Defect Detection\n"]
for title, text in SEC:
    md_lines.append(f"## {title}\n{text}\n")
    for cap, d in tables.get(title, []):
        md_lines.append(f"**{cap}**\n\n{tbl_md(d)}\n")
    for img, cap in images.get(title, []):
        md_lines.append(f"![{cap}](results_v2/{img})\n\n*{cap}*\n")
(RESULTS / "LAB05_Report_v2.md").write_text("\n".join(md_lines))

# ---- PDF version via PdfPages ----
def pdf_page(title, text, tbls=None, figs=None):
    fig = plt.figure(figsize=(8.27, 11.69)); fig.text(0.08, 0.95, title, fontsize=14, fontweight="bold")
    y = 0.90
    for i in range(0, len(text), 105):
        fig.text(0.08, y, text[i:i+105], fontsize=9.5); y -= 0.035
    y -= 0.01
    for cap, d in (tbls or []):
        hgt = 0.028 * (len(d) + 2)
        ax = fig.add_axes([0.08, max(0.06, y - hgt), 0.84, hgt]); ax.axis("off")
        t = ax.table(cellText=[[f"{v:.4f}" if isinstance(v, float) else str(v) for v in r] for r in d.values],
                     colLabels=list(d.columns), loc="center", cellLoc="center")
        t.auto_set_font_size(False); t.set_fontsize(7.5); t.scale(1, 1.15)
        fig.text(0.08, y, cap, fontsize=9, fontweight="bold"); y -= hgt + 0.04
    for img, cap in (figs or []):
        im = plt.imread(RESULTS / img); hgt = min(0.40, y - 0.08)
        ax = fig.add_axes([0.15, y - hgt, 0.70, hgt]); ax.imshow(im); ax.axis("off")
        fig.text(0.5, y - hgt - 0.015, cap, fontsize=8, ha="center", color="gray"); y -= hgt + 0.05
    return fig

with PdfPages(RESULTS / "LAB05_Report_v2.pdf") as pdf:
    cover = plt.figure(figsize=(8.27, 11.69))
    cover.text(0.5, 0.72, "LAB 05 v2", ha="center", fontsize=26, fontweight="bold")
    cover.text(0.5, 0.66, "HOG-Based Industrial Defect Detection and Classification", ha="center", fontsize=14)
    cover.text(0.5, 0.60, f"NEU steel dataset | final F1 {fin['F1']:.3f} | 6-class macro-F1 {mc_row.MacroF1:.3f}", ha="center", fontsize=10)
    cover.text(0.5, 0.40, f"HOG: {BEST_CELL}x{BEST_CELL} cells, {BEST_BIN} bins | SVM-RBF", ha="center", fontsize=10)
    pdf.savefig(cover); plt.close(cover)
    for title, text in SEC:
        pdf.savefig(pdf_page(title, text, tables.get(title), images.get(title))); plt.close("all")
print("saved:", RESULTS / "LAB05_Report_v2.pdf", "and", RESULTS / "LAB05_Report_v2.md")

## Summary of generated files (`results_v2/`)
| File | Content |
|---|---|
| `LAB05_Report_v2.pdf` / `.md` | 12-section report with this run's real numbers |
| `defect_model_v2.joblib` | trained final model bundle |
| `qc_realtime_v2.py` | standalone webcam/video prototype |
| `fig1…fig13 *.png` | dataset, patches, HOG visuals, comparison, confusion matrices, tuning heat-maps, robustness, QC & stream demos |

All lab requirements are covered: dataset inspection → resize/grayscale (v2 adds denoise + CLAHE) → HOG + visualisation → SVM & extra classifiers → comparison → confusion matrices & classification reports → accuracy/precision/recall/F1 → cell-size & orientation study → robustness under brightness/noise/rotation/blur → augmentation check → ACCEPT/REJECT decision module → report → webcam bonus.